# Helper functions


In [ ]:
import numpy as np 
import pandas as pd 
import os
import sys
import requests
import shutil
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
import kagglehub


print(" Installing dependencies and cloning repositories...")
!pip install -q timm yacs iopath kagglehub
!pip install -q git+https://github.com/fra31/auto-attack.git
!pip install -U safetensors


os.chdir('/content')
if os.path.exists('/content/SAR'): shutil.rmtree('/content/SAR')
if os.path.exists('/content/test-time-adaptation'): shutil.rmtree('/content/test-time-adaptation')

!git clone https://github.com/mr-eggplant/SAR.git /content/SAR
!git clone --depth 1 https://github.com/mariodoebler/test-time-adaptation.git /content/test-time-adaptation


sys.path.append('/content/SAR')
sys.path.insert(0, '/content/test-time-adaptation/classification')


import sar
def stable_softmax_entropy(x):
    return -(x.softmax(1) * x.log_softmax(1)).sum(1)
sar.softmax_entropy = stable_softmax_entropy


import types
import robustbench
try:
    import robustbench.loaders as rb_loaders
except ImportError:
    rb_loaders = types.ModuleType('loaders')
    robustbench.loaders = rb_loaders
    sys.modules['robustbench.loaders'] = rb_loaders

class DummyDataset(torch.utils.data.Dataset):
    def __init__(self, *args, **kwargs): pass
    def __len__(self): return 0
rb_loaders.CustomCifarDataset = DummyDataset
rb_loaders.CustomImageFolder = datasets.ImageFolder


In [ ]:
import torch, gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(" GPU Memory Cleared")


clear_gpu()

# ResNet Baseline


In [ ]:
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from tqdm import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")


data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")

data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

print(" Loading ResNet50-GN and native transforms...")
model = timm.create_model('resnet50_gn.a1h_in1k', pretrained=True).to(device)
model.eval()

config = resolve_data_config({}, model=model)
timm_transform = create_transform(**config)

print(f" Loading data from: {data_path}")
dataset = datasets.ImageFolder(root=data_path, transform=timm_transform)

print(" Remapping WordNet IDs to ImageNet classes...")
url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
mapping = requests.get(url).json()
wnid_to_idx = {v[0]: int(k) for k, v in mapping.items()}

new_samples = [(p, wnid_to_idx[os.path.basename(os.path.dirname(p))])
               for p, _ in dataset.samples if os.path.basename(os.path.dirname(p)) in wnid_to_idx]
dataset.samples = new_samples
dataset.targets = [s[1] for s in new_samples]

print(f" Dataset ready. Total images: {len(dataset)} (Should be exactly 50,000)")

dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

correct = 0
total = 0

print(" Running Base Evaluation...")
with torch.no_grad():
    for images, labels in tqdm(dataloader, desc="Evaluating", leave=True):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        pred = outputs.argmax(1)

        correct += (pred == labels).sum().item()
        total += labels.size(0)


final_accuracy = 100 * correct / total
print(f"\n Final Base Accuracy: {final_accuracy:.2f}%")

In [ ]:
clear_gpu() 


# Resnet + SAR + Attack


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
for obj in ['attack_model_resnet', 'sar_base_resnet', 'sar_model_resnet', 'optimizer_resnet', 'dataloader', 'dataset']:
    if obj in locals():
        del locals()[obj]

gc.collect()
torch.cuda.empty_cache()

print(f" VRAM Cleared: {torch.cuda.memory_allocated() / 1024**3:.2f} GB used.")

In [ ]:
import torch.nn.functional as F
import torch.optim as optim



from sar import SAR
from sam import SAM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")




print(" Initializing Greybox Attack Surrogate (Standard ResNet-50)...")
attack_model_resnet = timm.create_model('resnet50', pretrained=True).to(device).eval()
for param in attack_model_resnet.parameters():
    param.requires_grad = False


attack_model_resnet = timm.create_model('resnet50', pretrained=True).to(device).eval()
for param in attack_model_resnet.parameters():
    param.requires_grad = False


print(" Initializing TSMA Surrogate (Standard ResNet-50)...")
attack_model_resnet = timm.create_model('resnet50', pretrained=True).to(device).eval()
for param in attack_model_resnet.parameters():
    param.requires_grad = False
def tsma_attack_optimized(tta_model, images, labels, epsilon=8/255, alpha=2/255, steps=5, decay=1.0, lambda_drift=0.2):


    mean_std = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(images.device)
    std_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(images.device)
    images_unnorm = (images * std_std) + mean_std

    
    with torch.no_grad():
        clean_features = attack_model_resnet.forward_features(images)
        
        
        clean_mu = clean_features.mean(dim=(2, 3))
        clean_var = clean_features.var(dim=(2, 3))

    
    delta = torch.zeros_like(images_unnorm).uniform_(-epsilon, epsilon)
    delta = torch.clamp(images_unnorm + delta, min=0, max=1) - images_unnorm
    delta.requires_grad_(True)
    
    momentum = torch.zeros_like(delta).detach()

    
    with torch.enable_grad():
        for _ in range(steps):
            adv_images_unnorm = images_unnorm + delta
            adv_images = (adv_images_unnorm - mean_std) / std_std

            outputs = attack_model_resnet(adv_images)
            adv_features = attack_model_resnet.forward_features(adv_images)
            
            adv_mu = adv_features.mean(dim=(2, 3))
            adv_var = adv_features.var(dim=(2, 3))

            
            loss_ce_true = F.cross_entropy(outputs, labels)

            
            
            probs = F.softmax(outputs, dim=1)
            loss_entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=1).mean()

            
            loss_drift = F.mse_loss(adv_mu, clean_mu) + F.mse_loss(adv_var, clean_var)

            
            loss = -loss_ce_true + (1.5 * loss_entropy) - (lambda_drift * loss_drift)

            if delta.grad is not None:
                delta.grad.zero_()

            loss.backward()

            
            with torch.no_grad():
                grad = delta.grad
                
                
                grad_norm = torch.norm(grad.view(grad.shape[0], -1), p=1, dim=1).view(-1, 1, 1, 1)
                grad_norm = torch.clamp(grad_norm, min=1e-8)
                grad_normalized = grad / grad_norm
                
                momentum = decay * momentum + grad_normalized
                
                
                delta.data = delta.data - alpha * momentum.sign()
                delta.data = torch.clamp(delta.data, min=-epsilon, max=epsilon)
                delta.data = torch.clamp(images_unnorm + delta.data, min=0, max=1) - images_unnorm

    final_adv_images_unnorm = images_unnorm + delta.detach()
    return (final_adv_images_unnorm - mean_std) / std_std


print(" Initializing Target SAR Pipeline (ResNet-50-GN)...")
sar_base_resnet = timm.create_model('resnet50_gn.a1h_in1k', pretrained=True).to(device).float()

config = resolve_data_config({}, model=sar_base_resnet)
timm_transform = create_transform(**config)

print(f" Loading data from: {data_path}")
dataset = datasets.ImageFolder(root=data_path, transform=timm_transform)

print(" Remapping WordNet IDs to ImageNet classes...")
url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
mapping = requests.get(url).json()
wnid_to_idx = {v[0]: int(k) for k, v in mapping.items()}

new_samples = [(p, wnid_to_idx[os.path.basename(os.path.dirname(p))])
               for p, _ in dataset.samples if os.path.basename(os.path.dirname(p)) in wnid_to_idx]
dataset.samples = new_samples
dataset.targets = [s[1] for s in new_samples]

dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

def configure_model_and_params(model):

    model.train()
    model.requires_grad_(False)
    params = []

    for name, m in model.named_modules():
        
        if 'layer4' in name:
            continue

        if isinstance(m, (nn.BatchNorm2d, nn.LayerNorm, nn.GroupNorm)):
            m.requires_grad_(True)

            if isinstance(m, nn.BatchNorm2d):
                m.track_running_stats = False
                m.running_mean = None
                m.running_var = None

            for np_name, p in m.named_parameters():
                if np_name in ['weight', 'bias']:
                    params.append(p)

    return params

params_to_adapt = configure_model_and_params(sar_base_resnet)


margin_e0 = 0.4 * torch.log(torch.tensor(1000.0)).to(device)
optimizer_resnet = SAM(params_to_adapt, optim.SGD, lr=0.00025, momentum=0.9, rho=0.05)
sar_model_resnet = SAR(sar_base_resnet, optimizer=optimizer_resnet, steps=1, episodic=False, margin_e0=margin_e0)
correct_total, total_total = 0, 0
correct_clean, total_clean = 0, 0
correct_pois, total_pois = 0, 0

threshold = margin_e0.item()
POISON_FREQ = 2  

gc.collect()
torch.cuda.empty_cache()

pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="SAR ResNet-GN (Poisoned)")

for i, (images, labels) in pbar:
    images = images.to(device).float()
    labels = labels.to(device)

    
    is_poisoned = (i % POISON_FREQ == 0)

    
    if is_poisoned:
        inputs = tsma_attack_optimized(sar_base_resnet, images, labels)
    else:
        inputs = images

    outputs = sar_model_resnet(inputs)

    preds = outputs.argmax(dim=1)

    batch_correct = (preds == labels).sum().item()
    batch_total = labels.size(0)
    batch_acc = 100.0 * batch_correct / batch_total

    correct_total += batch_correct
    total_total += batch_total

    if is_poisoned:
        correct_pois += batch_correct
        total_pois += batch_total
    else:
        correct_clean += batch_correct
        total_clean += batch_total

    total_acc = 100.0 * correct_total / total_total
    clean_acc = 100.0 * correct_clean / max(1, total_clean)
    pois_acc = 100.0 * correct_pois / max(1, total_pois)

    
    probs = F.softmax(outputs, dim=1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=1)
    pass_rate = (entropy < threshold).float().mean().item()
    
    pbar.set_postfix({
        "Total": f"{total_acc:.2f}%",
        "Batch": f"{batch_acc:.2f}%",
        "Clean": f"{clean_acc:.2f}%",
        "Pois": f"{pois_acc:.2f}%",
        "Pass%": f"{pass_rate*100:.1f}",
        "Pois?": "Y" if is_poisoned else "N"
    })

print("\n FINAL RESULTS")
print(f"Total Accuracy : {100.0 * correct_total / total_total:.2f}%")
print(f"Clean Accuracy : {100.0 * correct_clean / max(1, total_clean):.2f}%")
print(f"Poison Accuracy: {100.0 * correct_pois / max(1, total_pois):.2f}%")